# KEEPCODING · AI Engineering · M1 · Agentes con LangChain

## Clase 5 · Lab 3 · Crea y consume servidores MCP

En este laboratorio vas a **construir un servidor MCP desde cero**, consumirlo desde un agente LangChain y, sobre todo, **componer tu servidor con un servidor MCP público real** (DeepWiki) en el mismo agente. Todo dentro del mismo notebook.

**MCP** (*Model Context Protocol*) es un estándar abierto para conectar agentes con herramientas y datos. La gracia: una tool que escribes una vez en un servidor MCP la puede usar este notebook, Claude Desktop, un IDE o cualquier otro cliente MCP. Y al revés: **tu cliente puede consumir cualquier servidor MCP del mundo** sin escribir ni una línea de integración.

El servidor que construirás expone una tool de **One Piece** que consulta la API en vivo (con fallback local). El servidor público que consumirás es **DeepWiki MCP**: expone tools para consultar la documentación de cualquier repositorio de GitHub.

### Qué necesitas

- Una clave **`OPENAI_API_KEY`** (el agente usa el modelo `gpt-5-mini`).
- Corre igual en **Google Colab** o en **Jupyter local**. El notebook se ejecuta de arriba a abajo.
- Conexión a internet: el agente llama a OpenAI, a la API de One Piece y al servidor público de DeepWiki.

### A quién va dirigido

- Alumnado del módulo M1 que ya ha hecho los Labs 1 y 2 (agente ReAct y LangGraph) y quiere entender de dónde salen las tools cuando no están en su código.

### Por qué importa

MCP es el **USB-C de las tools**: escribes una integración una vez y la consume cualquier cliente (este notebook, Claude Desktop, un IDE). Es el mismo protocolo que usan los harness de M4 (OpenClaw) y M5 (Harness Engineering) para cargar capacidades.

### Conecta con

- **Slides M1** · MCP, cliente y servidor.
- **Clase 4 / M3** · RAG agéntico con el MCP público context7.
- **M5** · MCP aparece en los protocolos soportados del `AGENTS.md`.
- **Labs 1-2** · de dónde salen las tools cuando no están en tu código.

### Stack

`mcp` (FastMCP) · `langchain-mcp-adapters` · `langchain>=1.0` · `gpt-5-mini` · transporte `streamable_http`.

### Hardware

Cualquier CPU; Colab Free sobra. El servidor MCP corre en un hilo del propio notebook.


## Guion del laboratorio

1. **Setup**: instalar dependencias y cargar la API key.
2. **Repaso**: los tres roles de MCP (host, client, server).
3. **Construir el servidor**: definir un servidor `FastMCP` con una tool.
4. **Arrancar y conectar**: levantar el servidor por HTTP y conectar el `MultiServerMCPClient`.
5. **Cargar las tools**: `get_tools()` convierte las tools MCP en tools de LangChain.
6. **Construir el agente**: `create_agent` con esas tools MCP.
7. **Demo**: el agente responde una pregunta de One Piece usando una tool de TU servidor MCP.
8. **Componer con otros servidores MCP**: añades el servidor público DeepWiki y el agente pasa a tener tools de los DOS.
9. **Ejercicio**.
10. **Recap y retos**.

## 1. Setup

### 1.1. Instalar dependencias

Instalamos lo necesario para el servidor MCP y para el agente:

- `mcp`: el SDK oficial de MCP en Python, incluye `FastMCP` para crear servidores.
- `langchain-mcp-adapters`: el puente que convierte tools MCP en tools de LangChain.
- `langchain-openai` y `langgraph`: el modelo y el agente ReAct ya construido.

> En Colab esto tarda unos segundos. La opción `-q` reduce el ruido en la salida.

In [1]:
# Instalación de dependencias (ejecutar una sola vez por sesión)
%pip install -q mcp langchain-mcp-adapters "langchain-openai" "langchain>=1.0" "langgraph>=1.0"

### 1.2. Cargar la API key de OpenAI

El agente necesita `OPENAI_API_KEY`. Esta celda intenta tres fuentes en orden:

1. La variable de entorno (si ya la tienes configurada).
2. El gestor de secretos de Google Colab (`google.colab.userdata`).
3. Un cuadro de entrada seguro con `getpass` (no muestra la clave en pantalla).

Así el notebook funciona tanto en Colab como en Jupyter local sin tocar el código.

In [2]:
import os
from getpass import getpass


def cargar_api_key() -> None:
    """Asegura que OPENAI_API_KEY esté disponible en el entorno."""
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY encontrada en las variables de entorno.")
        return

    # Intento 2: secretos de Google Colab.
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("OPENAI_API_KEY cargada desde los secretos de Colab.")
        return
    except Exception:
        pass

    # Intento 3: entrada manual segura.
    os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OPENAI_API_KEY: ")
    print("OPENAI_API_KEY cargada desde la entrada manual.")


cargar_api_key()

OPENAI_API_KEY encontrada en las variables de entorno.


## 2. Repaso: los tres roles de MCP

MCP separa el trabajo en tres piezas. Conviene tenerlas claras antes de escribir código:

| Rol | Quién es | Qué hace |
|-----|----------|----------|
| **Host** | La aplicación con el modelo (este notebook, Claude Desktop, un IDE) | Decide cuándo usar herramientas y orquesta la conversación. |
| **Client** | El conector dentro del host | Habla el protocolo MCP, abre una conexión por cada servidor. |
| **Server** | Tu proceso de herramientas | Expone tools, recursos y prompts. Vive aparte y puede reutilizarse. |

La idea clave: el **servidor es independiente**. Hoy lo consume este notebook; mañana el mismo servidor lo puede consumir Claude Desktop o un IDE.

Sobre el **transporte**: MCP soporta varios. El clásico es `stdio` (el cliente lanza el servidor como subproceso por entrada/salida estándar), que usan apps de escritorio como Claude Desktop. Dentro de un notebook ese transporte da problemas, así que aquí usamos el transporte **HTTP** (`streamable-http`): el servidor escucha en un puerto y el cliente se conecta por URL. El código del servidor y de las tools es idéntico: solo cambia como se arranca.

## 3. Construir el servidor MCP

Definimos el servidor en esta celda con `FastMCP`. Anatomía:

- `mcp = FastMCP("onepiece")`: crea el servidor y le pone nombre.
- `mcp.settings.host` y `mcp.settings.port`: dónde escuchará (`127.0.0.1:8765`).
- `@mcp.tool()`: decora una función para exponerla como tool. El **docstring** y los **tipos** se convierten en el esquema que verá el agente.

La tool `buscar_personaje` intenta la API en vivo de One Piece (descarga la lista de personajes una vez y la filtra) y, si falla, cae a un diccionario local. Así la clase no depende de la red.

In [3]:
"""Servidor MCP de One Piece.

Define una tool, 'buscar_personaje', que devuelve la recompensa (bounty)
de un personaje. Lo consume este notebook y lo podría consumir cualquier
otro cliente MCP.
"""
import urllib.request
import json
import unicodedata

from mcp.server.fastmcp import FastMCP

# Creamos el servidor MCP, le damos nombre y configuramos dónde escuchará.
mcp = FastMCP("onepiece")
mcp.settings.host = "127.0.0.1"
mcp.settings.port = 8765
mcp.settings.log_level = "WARNING"  # menos ruido de logs en la salida del notebook

# Silenciamos los logs informativos de uvicorn, httpx y del SDK de MCP.
import logging

logging.getLogger().setLevel(logging.WARNING)
for _logger in ("uvicorn", "uvicorn.access", "uvicorn.error", "httpx", "httpx2", "mcp"):
    logging.getLogger(_logger).setLevel(logging.WARNING)

# Cache en memoria: la lista completa de personajes se descarga una sola vez.
_OP_CACHE = []

# Plan B: recompensas canónicas post-Wano de los Sombrero de Paja (en berries).
RECOMPENSAS_FALLBACK = {
    "luffy": 3_000_000_000,
    "zoro": 1_111_000_000,
    "sanji": 1_032_000_000,
    "jinbe": 1_100_000_000,
    "nami": 366_000_000,
    "usopp": 500_000_000,
    "brook": 383_000_000,
    "franky": 394_000_000,
    "robin": 930_000_000,
    "chopper": 1_000,
}


def _normalizar(texto: str) -> str:
    """Minúsculas, sin acentos ni puntuación, espacios colapsados."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = "".join(c if c.isalnum() else " " for c in t.lower())
    return " ".join(t.split())


def _coincide(consulta: str, nombre: str) -> bool:
    """True si 'consulta' identifica a 'nombre'. Tolera puntos, acentos y orden.

    El LLM puede pasar "Luffy" o "Monkey D. Luffy"; la API guarda
    "Monkey D Luffy" (sin punto). Normalizamos ambos y comparamos por
    contención o por palabras, nunca con un substring frágil.
    """
    q, n = _normalizar(consulta), _normalizar(nombre)
    if not q or not n:
        return False
    if q == n or q in n or n in q:
        return True
    palabras = [w for w in q.split() if len(w) >= 3]
    return bool(palabras) and all(w in n.split() for w in palabras)


def _cargar_personajes_op():
    """Descarga (una sola vez) la lista completa de personajes de One Piece."""
    global _OP_CACHE
    if _OP_CACHE:
        return _OP_CACHE
    url = "https://api.api-onepiece.com/v2/characters/en"
    req = urllib.request.Request(url, headers={"User-Agent": "keepcoding-ai-lab/1.0"})
    with urllib.request.urlopen(req, timeout=25) as r:
        _OP_CACHE = json.load(r)
    return _OP_CACHE


@mcp.tool()
def buscar_personaje(nombre: str) -> str:
    """Busca la recompensa (bounty) de un personaje de One Piece por su nombre.

    Intenta primero la API en vivo de One Piece y, si falla, usa una base de
    datos local con las recompensas canónicas de la tripulación de Sombrero
    de Paja. Devuelve un texto con el nombre y la recompensa en berries.
    """
    # Intento 1: API en vivo (lista completa cacheada, se filtra en Python).
    try:
        for c in _cargar_personajes_op():
            if _coincide(nombre, c.get("name") or ""):
                return (
                    f"{c['name']} tiene una recompensa de "
                    f"{c.get('bounty') or 'desconocida'} berries (fuente: API One Piece)."
                )
        return f"No se ha encontrado a '{nombre}' en la API de One Piece."
    except Exception:
        # Cualquier fallo de red o de datos nos lleva al plan B.
        pass

    # Intento 2: base de datos local.
    for personaje, recompensa in RECOMPENSAS_FALLBACK.items():
        if _coincide(nombre, personaje):
            return (
                f"{personaje.capitalize()} tiene una recompensa de "
                f"{recompensa} berries (fuente: base de datos local)."
            )

    return f"No se ha encontrado a '{nombre}' ni en la API ni en la base local."

El servidor ya está definido. Ahora lo **arrancamos** en un hilo de fondo para que escuche peticiones por HTTP sin bloquear el notebook. Es el mismo patrón que usarás en el lab de A2A (M3) para levantar un servidor dentro del notebook.

In [4]:
import threading
import time


def _arrancar_servidor():
    """Arranca el servidor MCP por HTTP. Corre en un hilo de fondo."""
    mcp.run(transport="streamable-http")


# Lo lanzamos en un hilo daemon para no bloquear el notebook.
# El guard evita arrancar dos veces si reejecutas la celda.
if not globals().get("_servidor_mcp_arrancado"):
    threading.Thread(target=_arrancar_servidor, daemon=True).start()
    globals()["_servidor_mcp_arrancado"] = True
    time.sleep(4)  # margen para que el servidor termine de levantar
    print("Servidor MCP escuchando en http://127.0.0.1:8765/mcp")
else:
    print("El servidor MCP ya estaba arrancado.")

Servidor MCP escuchando en http://127.0.0.1:8765/mcp


## 4. Conectar el cliente MCP

El servidor ya escucha en `http://127.0.0.1:8765/mcp`. Ahora toca el lado del cliente.

El cliente y el servidor se hablan de forma **asíncrona** (`async`/`await`). En Colab y en Jupyter puedes usar `await` directamente dentro de una celda, así que no necesitas nada especial para ello.

`MultiServerMCPClient` recibe un diccionario con un servidor por entrada. Para nuestro server por HTTP indicamos:

- `transport`: `"streamable_http"`.
- `url`: la dirección donde escucha el servidor.

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Configuramos el cliente apuntando a nuestro servidor de One Piece por HTTP.
cliente_mcp = MultiServerMCPClient(
    {
        "onepiece": {
            "transport": "streamable_http",
            "url": "http://127.0.0.1:8765/mcp",
        }
    }
)

print("Cliente MCP configurado. Servidor objetivo: http://127.0.0.1:8765/mcp")

Cliente MCP configurado. Servidor objetivo: http://127.0.0.1:8765/mcp


## 5. Cargar las tools del MCP

`get_tools()` es el momento de la verdad: el cliente se conecta al servidor, hace el *handshake* MCP, le pregunta qué tools expone y las devuelve **ya convertidas en tools de LangChain**.

Es una llamada `async`, por eso usamos `await`. Imprimimos qué tools han llegado para verlo con nuestros propios ojos.

In [6]:
# get_tools() es async: el cliente se conecta al servidor y negocia el protocolo.
herramientas_mcp = await cliente_mcp.get_tools()

print(f"Tools recibidas del servidor MCP: {len(herramientas_mcp)}\n")
for herramienta in herramientas_mcp:
    print(f"- {herramienta.name}: {herramienta.description.strip().splitlines()[0]}")

Tools recibidas del servidor MCP: 1

- buscar_personaje: Busca la recompensa (bounty) de un personaje de One Piece por su nombre.


Cada elemento de `herramientas_mcp` es una tool de LangChain normal y corriente. La diferencia es que **no ejecuta código local**: cuando el agente la invoque, el adaptador enviará la petición al servidor MCP y esperará la respuesta.

## 6. Construir el agente con las tools MCP

Las tools ya están en formato LangChain, así que el agente se construye exactamente igual que en los labs anteriores. Usamos `create_agent` con el modelo `gpt-5-mini` y le pasamos `herramientas_mcp`.

El detalle importante: el agente **no sabe ni le importa** que sus tools vienen de un servidor MCP en otro proceso. Para él son tools normales.

In [7]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # LangChain v1.0+ · sustituye al deprecado create_react_agent

# El modelo de chat. gpt-5-mini es rápido y barato para clase.
modelo = ChatOpenAI(model="gpt-5-mini")

# Instrucciones de sistema para el agente.
PROMPT_SISTEMA = (
    "Eres un experto en One Piece especializado en recompensas (bounties). "
    "Usa la tool 'buscar_personaje' para obtener la recompensa de cada "
    "personaje: no respondas de memoria. Contesta en español, claro y con "
    "las cifras bien formateadas."
)

# El agente se construye igual que siempre: las tools MCP son tools normales.
agente = create_agent(
    model=modelo,
    tools=herramientas_mcp,
    system_prompt=PROMPT_SISTEMA,
)

print("Agente ReAct construido con las tools del servidor MCP.")

Agente ReAct construido con las tools del servidor MCP.


## 7. Demo: el agente usa una tool que vive en el servidor MCP

Momento estrella. Le hacemos al agente una pregunta de One Piece.

Fíjate en lo que pasa por detrás: el agente decide llamar a `buscar_personaje`, pero esa función no está en el código del agente. Vive en el **servidor MCP**, en un hilo aparte que escucha por HTTP. El adaptador MCP hace de mensajero: lleva la petición, espera, y trae el resultado.

In [8]:
# Pregunta para el agente. La tool que responderá vive en el servidor MCP.
pregunta = "¿Cuál es la recompensa de Zoro? ¿Y la de Nami?"

resultado = await agente.ainvoke(
    {"messages": [{"role": "user", "content": pregunta}]}
)

# El último mensaje contiene la respuesta final del agente.
print("Pregunta:", pregunta)
print("\nRespuesta del agente:")
print(resultado["messages"][-1].content)

Pregunta: ¿Cuál es la recompensa de Zoro? ¿Y la de Nami?

Respuesta del agente:
Roronoa Zoro, 1.101.000.000 berries (según la API de One Piece).  
Nami, 366.000.000 berries (según la API de One Piece).  

¿Quieres la recompensa de algún otro personaje?


Para verlo aún más claro, recorremos todos los mensajes del intercambio. Verás la llamada a la tool (`tool_calls`) y el mensaje de tipo `tool` con lo que devolvió el servidor MCP.

In [9]:
# Recorremos la traza completa: pregunta, llamada a tool, respuesta del MCP, respuesta final.
for mensaje in resultado["messages"]:
    tipo = mensaje.__class__.__name__
    if getattr(mensaje, "tool_calls", None):
        for llamada in mensaje.tool_calls:
            print(f"[{tipo}] llama a tool '{llamada['name']}' con args {llamada['args']}")
    elif mensaje.text:
        # .text extrae el texto aunque el contenido venga en bloques (tools MCP).
        print(f"[{tipo}] {mensaje.text}")

[HumanMessage] ¿Cuál es la recompensa de Zoro? ¿Y la de Nami?
[AIMessage] llama a tool 'buscar_personaje' con args {'nombre': 'Roronoa Zoro'}
[AIMessage] llama a tool 'buscar_personaje' con args {'nombre': 'Nami'}
[ToolMessage] Roronoa Zoro tiene una recompensa de 1.101.000.000 berries (fuente: API One Piece).
[ToolMessage] Nami tiene una recompensa de 366.000.000 berries (fuente: API One Piece).
[AIMessage] Roronoa Zoro, 1.101.000.000 berries (según la API de One Piece).  
Nami, 366.000.000 berries (según la API de One Piece).  

¿Quieres la recompensa de algún otro personaje?


## 8. Componer con otros servidores MCP

Tu agente ya consume tu propio servidor. Pero la **gracia real de MCP** no es esa: es que un mismo cliente puede hablar con **varios servidores a la vez**, los suyos y los de otros, sin escribir ni una línea de integración por servidor.

Lo demostramos con un servidor MCP **real, público y gratis**: [DeepWiki MCP](https://mcp.deepwiki.com/). Expone tres tools para consultar la documentación de cualquier repositorio de GitHub: `read_wiki_structure`, `read_wiki_contents` y `ask_question`.

Reconfiguramos el `MultiServerMCPClient` con **dos** servidores: el tuyo (`onepiece`) y el público (`deepwiki`). El cliente hace los dos handshakes MCP y nos devuelve todas las tools juntas.

In [10]:
# Cliente MCP apuntando a DOS servidores: el tuyo (local) y uno público real.
cliente_multi = MultiServerMCPClient(
    {
        "onepiece": {
            "transport": "streamable_http",
            "url": "http://127.0.0.1:8765/mcp",
        },
        "deepwiki": {
            "transport": "streamable_http",
            "url": "https://mcp.deepwiki.com/mcp",
        },
    }
)

# get_tools() ahora consulta los DOS servidores y junta todas las tools.
herramientas_multi = await cliente_multi.get_tools()

print(f"Tools recibidas de ambos servidores: {len(herramientas_multi)}\n")
for t in herramientas_multi:
    primera_linea = (t.description or "").strip().splitlines()[0]
    print(f"  - {t.name}: {primera_linea[:80]}")

Tools recibidas de ambos servidores: 4

  - buscar_personaje: Busca la recompensa (bounty) de un personaje de One Piece por su nombre.
  - ask_wiki_question: Ask any question about a GitHub repository's codebase and get an AI-powered answ
  - read_wiki_contents: View documentation about a GitHub repository.
  - read_wiki_structure: Get a list of documentation topics for a GitHub repository.


Cuatro tools en total: una tuya (`buscar_personaje`) y tres de DeepWiki. Para el agente son **todas iguales**: tools de LangChain con su schema. Le da igual que vivan en procesos distintos, en máquinas distintas o que las haya escrito otro equipo.

Reconstruimos el agente con el conjunto completo y le damos un *system prompt* que sepa **cuándo usar cada una**.

In [11]:
# Mismo modelo, pero ahora con tools de los DOS servidores MCP.
PROMPT_MULTI = (
    "Eres un asistente con dos competencias muy distintas:\n"
    "- Sobre One Piece: usa 'buscar_personaje' para obtener la recompensa "
    "de un personaje desde la API en vivo.\n"
    "- Sobre repositorios de código en GitHub: usa las tools de DeepWiki "
    "('read_wiki_structure', 'read_wiki_contents', 'ask_question') para "
    "responder consultando la documentación del repo.\n"
    "Elige la tool según la pregunta. Responde en español, claro y conciso."
)

agente_multi = create_agent(
    model=modelo,
    tools=herramientas_multi,
    system_prompt=PROMPT_MULTI,
)

print(f"Agente reconstruido con {len(herramientas_multi)} tools de 2 servidores MCP.")

Agente reconstruido con 4 tools de 2 servidores MCP.


### Demo: el agente elige el servidor MCP correcto

Dos preguntas pensadas para disparar **un servidor distinto cada una**. Mira la traza: en la primera el agente llama a una tool de tu servidor; en la segunda, a una tool del servidor público de DeepWiki. El agente no sabe nada de "tu servidor" o "el de DeepWiki": solo elige la tool que mejor encaja con la pregunta.

In [12]:
# Pregunta 1: dispara la tool de TU servidor (onepiece).
pregunta_1 = "¿Cuál es la recompensa de Sanji?"
print(f"PREGUNTA 1: {pregunta_1}")
print("-" * 65)
res_1 = await agente_multi.ainvoke(
    {"messages": [{"role": "user", "content": pregunta_1}]}
)
for m in res_1["messages"]:
    if getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"  [tool call] {tc['name']}({tc['args']})")
    elif m.__class__.__name__ == "ToolMessage":
        print(f"  [tool result] {m.text[:120]}...")
print(f"\nRESPUESTA: {res_1['messages'][-1].content}\n")

# Pregunta 2: dispara las tools de DeepWiki (servidor público externo).
pregunta_2 = (
    "Consultando la documentación del repo 'modelcontextprotocol/python-sdk', "
    "explica brevemente qué es un FastMCP."
)
print("=" * 65)
print(f"PREGUNTA 2: {pregunta_2}")
print("-" * 65)
res_2 = await agente_multi.ainvoke(
    {"messages": [{"role": "user", "content": pregunta_2}]}
)
for m in res_2["messages"]:
    if getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"  [tool call] {tc['name']}({tc['args']})")
    elif m.__class__.__name__ == "ToolMessage":
        print(f"  [tool result] {m.text[:120]}...")
print(f"\nRESPUESTA: {res_2['messages'][-1].content}")

PREGUNTA 1: ¿Cuál es la recompensa de Sanji?
-----------------------------------------------------------------


  [tool call] buscar_personaje({'nombre': 'Sanji'})
  [tool result] Sanji tiene una recompensa de 1.032.000.000 berries (fuente: API One Piece)....

RESPUESTA: Sanji tiene una recompensa de 1.032.000.000 berries.

PREGUNTA 2: Consultando la documentación del repo 'modelcontextprotocol/python-sdk', explica brevemente qué es un FastMCP.
-----------------------------------------------------------------


  [tool call] read_wiki_contents({'repoName': 'modelcontextprotocol/python-sdk'})
  [tool result] # Page: Overview

# Overview

<details>
<summary>Relevant source files</summary>

The following files were used as conte...

RESPUESTA: FastMCP (ahora renombrado a MCPServer) es el framework de servidor de alto nivel del SDK: una API ergonómica y basada en decoradores para exponer herramientas, recursos y prompts del protocolo MCP sin escribir la lógica de bajo nivel.  

Características principales (resumen):
- API decorador: @mcp.tool(), @mcp.resource(), @mcp.prompt() para registrar handlers.  
- Generación automática de JSON Schema a partir de las anotaciones de tipos de Python (validación de entradas/salidas).  
- Inyección de Context en handlers (logging, progreso, lectura de recursos) y soporte de lifespan (setup/teardown de recursos compartidos).  
- Abstracción de transportes: delega al servidor de bajo nivel y funciona sobre stdio, SSE o StreamableHTTP.  
- Diseñado como el punto d

## 9. Ejercicio

Añade una **segunda tool** al servidor MCP: `comparar_recompensas(nombre_a, nombre_b)` que diga cuál de los dos personajes tiene la recompensa más alta.

Pistas:

1. Edita la celda del servidor (sección 3) y añade otra función decorada con `@mcp.tool()`.
2. Reinicia el kernel y ejecuta el notebook de arriba a abajo (así el servidor vuelve a arrancar con la tool nueva).
3. Comprueba con `get_tools()` que ahora llegan **dos** tools.

La siguiente celda es un andamiaje de la solución para que la copies dentro de la celda del servidor.

In [13]:
# Andamiaje de solución: este bloque va DENTRO de la celda del servidor MCP
# (sección 3), justo después de la tool buscar_personaje.
#
# @mcp.tool()
# def comparar_recompensas(nombre_a: str, nombre_b: str) -> str:
#     """Compara las recompensas de dos personajes y dice cuál es mayor."""
#     a = RECOMPENSAS_FALLBACK.get(nombre_a.strip().lower())
#     b = RECOMPENSAS_FALLBACK.get(nombre_b.strip().lower())
#     if a is None or b is None:
#         return "Alguno de los personajes no está en la base local."
#     if a == b:
#         return f"{nombre_a} y {nombre_b} tienen la misma recompensa."
#     ganador = nombre_a if a > b else nombre_b
#     return f"{ganador} tiene la recompensa más alta."
print("Copia el bloque comentado dentro de la celda del servidor y reejecuta el notebook.")

Copia el bloque comentado dentro de la celda del servidor y reejecuta el notebook.


## 10. Recap y retos

**Qué has hecho en este lab:**

- Has definido un **servidor MCP** desde cero con `FastMCP`, con una tool real y robusta.
- Lo has **arrancado por HTTP** en un hilo de fondo del notebook.
- Has conectado un **cliente** (`MultiServerMCPClient`) que se comunica con él por el transporte `streamable-http`.
- Has convertido las tools MCP en tools de LangChain con `get_tools()`.
- Has montado un agente ReAct que usa una herramienta que **no está en su código**, sino en el servidor MCP.
- Has **compuesto** tu servidor con un servidor MCP **público real** (DeepWiki) en el mismo agente. Cuatro tools de dos procesos distintos, una sola decisión del agente.

**La idea que te llevas:** un servidor MCP es independiente y reutilizable. El mismo servidor lo puede consumir **Claude Desktop**, un **IDE** con soporte MCP o **cualquier otro agente** compatible. Y al revés: tu cliente puede consumir cualquier servidor público (DeepWiki, GitHub MCP, etc.) sin escribir integraciones. Eso es lo que aporta el estándar MCP: **escribe una vez, conecta a cualquier cosa**.

**Retos para seguir:**

1. **Añade un recurso MCP.** Además de tools, MCP soporta *resources*. Expón uno con `@mcp.resource("crew://strawhats")` que devuelva la lista completa de la tripulación.
2. **Cambia el repo.** Pregunta al agente sobre otro repo que conozcas (por ejemplo `langchain-ai/langgraph`). DeepWiki indexa la mayoría de repos públicos al vuelo.
3. **Añade un tercer servidor MCP.** Hay [muchos públicos](https://github.com/modelcontextprotocol/servers). Encuentra uno que no requiera API key y mete sus tools en `MultiServerMCPClient`.
4. **Prueba el transporte stdio.** Fuera del notebook, en un script `.py`, arranca el servidor con `mcp.run(transport="stdio")`: es el transporte que usan por defecto las apps de escritorio como Claude Desktop.